# JEPA-Quant — VICReg Training

Trains the multimodal **Joint-Embedding Predictive Architecture** on the underlying
price series. All logic lives in `src/jepa_quant/` — this notebook only wires data,
config, and the training loop together.

**Anti-collapse:** VICReg (variance + covariance, invariance term dropped). The
regularizer is pluggable — set `cfg.regularizer = 'codebook'` to compare the soft
codebook bottleneck without changing any other code.

**Layout:** data + this notebook live in Google Drive; `src/` is pulled fresh from
GitHub on every run, so pushing from VS Code updates the model code here
automatically (reopen the notebook tab to refresh the notebook itself).

## 0 · Environment — mount Drive, pull latest `src/` from git, put it on the path

In [ ]:
import os, subprocess, sys
from pathlib import Path

# VS Code (local kernel) inherits VSCODE_* — treat as local dev, never clone/mount.
IN_VSCODE = bool(os.environ.get('VSCODE_PID') or os.environ.get('VSCODE_CWD'))
IN_COLAB  = ('google.colab' in sys.modules or os.path.exists('/content')) and not IN_VSCODE

REPO = 'shreyasnat2804/JEPA-quant'

if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount('/content/drive')

    # Pull the latest repo code (src/*.py) into the VM. This does NOT refresh the
    # notebook tab you're viewing — reopen via File -> Open notebook -> GitHub for that.
    token = userdata.get('GITHUB_TOKEN')
    dest  = '/content/JEPA-quant'
    if not os.path.exists(dest):
        subprocess.run(['git', 'clone', f'https://{token}@github.com/{REPO}.git', dest], check=True)
    else:
        subprocess.run(['git', '-C', dest, 'pull'], check=True)

    SRC      = f'{dest}/src'
    DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/JEPA-QUANT/data/raw/stocks'
else:
    root = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
    SRC      = str(root / 'src')
    DATA_DIR = str(root / 'data' / 'raw' / 'stocks')
    print(f"Running in {'VS Code' if IN_VSCODE else 'local'} — no Drive mount / git pull")

if SRC not in sys.path:
    sys.path.insert(0, SRC)
print('src on path :', SRC)
print('data dir    :', DATA_DIR)

In [ ]:
# --- Verify the VM clone is current (run right after the setup cell) ---
# Stale VM code is a recurring cause of running old defaults: e.g. the moirai
# price-encoder default was replaced by 'transformer' in commit ae39809. If `git
# pull` didn't reach the VM, you silently run the OLD code. This prints the loaded
# commit and flags drift so you catch it before training.
if IN_COLAB:
    log = subprocess.run(
        ['git', '-C', dest, 'log', '--oneline', '-1'],
        capture_output=True, text=True,
    ).stdout.strip()
    print('VM HEAD     :', log)
    # ae39809 = the commit that flipped the price-encoder default to 'transformer'.
    has_flip = subprocess.run(
        ['git', '-C', dest, 'merge-base', '--is-ancestor', 'ae39809', 'HEAD'],
    ).returncode == 0
    print('transformer default present :', has_flip)
    if not has_flip:
        print('!! STALE CLONE — re-run the setup cell above; `git pull` did not '
              'reach ae39809. Until it does you are running old src/ code.')
else:
    print('local / VS Code — using working-tree src, no VM clone to check')


In [ ]:
# Hot-reload edited jepa_quant/*.py modules without restarting the kernel.
# After you push from VS Code and re-run the setup cell's `git pull`, autoreload
# swaps the new code in on the next cell execution — no kernel restart, no lost state.
#
# Colab's bundled autoreload extension still does `from imp import reload`, but `imp`
# was removed in Python 3.12. Shim it with importlib.reload so the extension loads.
# No-op on Python <=3.11 (where `imp` exists) and on local VS Code.
import sys, types, importlib
if 'imp' not in sys.modules:
    _imp = types.ModuleType('imp')
    _imp.reload = importlib.reload
    sys.modules['imp'] = _imp
%load_ext autoreload
%autoreload 2

## 1 · Dependencies

`uni2ts` (Moirai) and `peft`/`transformers` (Qwen + LoRA) are only needed for the
foundation-model backends. The lightweight transformer backends run on torch alone.

In [ ]:
# --- Colab install (transformer backend, native numpy 2) ---
# The price encoder uses the lightweight built-in transformer backend, so we
# do NOT install uni2ts/Moirai. uni2ts forces numpy<2, which is incompatible
# with Colab's numpy-2-built torch/pandas/pyarrow/scipy and caused a cascade
# of C-ABI / numpy-corruption failures. Staying on Colab's native numpy 2
# keeps the whole stack consistent — no pins, no restart needed.
# (To run the real Moirai backend, do it in a clean numpy<2 env, not Colab.)
%pip install -q "transformers>=4.44" "peft>=0.11" accelerate einops matplotlib pyarrow

import numpy as _np
print(f"numpy: {_np.__version__}  (expected 2.x — Colab native)")


## 2 · Hugging Face login

Add `HF_TOKEN` in the Colab Secrets (🔑) tab. Required for Moirai, Qwen2.5, and FinBERT.

## 3 · Configuration

`USE_FOUNDATION_MODELS=True` → transformer price encoder + **Qwen2.5-1.5B + LoRA**
predictor (needs a GPU + HF access). Set it `False` for a fast CPU smoke run with the
lightweight transformer predictor. Everything else is identical.

> **Moirai is shelved on Colab.** Its `uni2ts` dependency forces numpy<2, which is
> incompatible with Colab's numpy-2-built torch/pandas/pyarrow stack and caused a
> cascade of C-ABI failures. The price encoder therefore uses the built-in
> `transformer` backend here. Run `backend='moirai'` only in a dedicated numpy<2
> environment — see the CLAUDE.md 2026-06-04 log.

import dataclasses as dc
import torch
from jepa_quant import JEPAConfig, registered_regularizers

print('Available anti-collapse methods:', registered_regularizers())

USE_FOUNDATION_MODELS = True   # Qwen2.5-1.5B+LoRA predictor (GPU). False = all-transformer CPU smoke run.
REGULARIZER           = 'vicreg'   # swap to 'codebook' to compare — nothing else changes.
CONTEXT_LENGTH        = 64
HORIZON               = 16
LATENT_DIM            = 256

cfg = JEPAConfig()
cfg = dc.replace(cfg, regularizer=REGULARIZER)
cfg = dc.replace(cfg, data=dc.replace(cfg.data, data_dir=DATA_DIR,
                                      context_length=CONTEXT_LENGTH, horizon=HORIZON))

if USE_FOUNDATION_MODELS:
    # Predictor = Qwen2.5-1.5B + LoRA (Option B, preferred). The price encoder stays
    # on the 'transformer' backend: Moirai is shelved on Colab because its uni2ts dep
    # forces numpy<2, which is incompatible with Colab's numpy-2-built stack. Run
    # backend='moirai' only in a dedicated numpy<2 env. See CLAUDE.md 2026-06-04 log.
    cfg = dc.replace(cfg,
        price_encoder=dc.replace(cfg.price_encoder, backend='transformer',
                                 context_length=CONTEXT_LENGTH, latent_dim=LATENT_DIM,
                                 freeze_backbone=False),
        predictor=dc.replace(cfg.predictor, backend='lm', latent_dim=LATENT_DIM))
    device, batch = ('cuda' if torch.cuda.is_available() else 'cpu'), 256
else:
    cfg = dc.replace(cfg,
        price_encoder=dc.replace(cfg.price_encoder, backend='transformer',
                                 context_length=CONTEXT_LENGTH, latent_dim=LATENT_DIM,
                                 freeze_backbone=False),
        predictor=dc.replace(cfg.predictor, backend='transformer', latent_dim=LATENT_DIM))
    device, batch = 'cpu', 64

cfg = dc.replace(cfg, train=dc.replace(cfg.train, device=device, batch_size=batch,
                                       max_steps=2000, warmup_proj_steps=500, num_workers=2))
print(f"backend(price)={cfg.price_encoder.backend}  backend(pred)={cfg.predictor.backend}"
      f"  reg={cfg.regularizer}  device={cfg.train.device}  batch={cfg.train.batch_size}")

In [ ]:
import dataclasses as dc
import torch
from jepa_quant import JEPAConfig, registered_regularizers

print('Available anti-collapse methods:', registered_regularizers())

USE_FOUNDATION_MODELS = True   # Moirai + Qwen2.5-1.5B+LoRA. Set False for a CPU smoke run.
REGULARIZER           = 'vicreg'   # swap to 'codebook' to compare — nothing else changes.
CONTEXT_LENGTH        = 64
HORIZON               = 16
LATENT_DIM            = 256

cfg = JEPAConfig()
cfg = dc.replace(cfg, regularizer=REGULARIZER)
cfg = dc.replace(cfg, data=dc.replace(cfg.data, data_dir=DATA_DIR,
                                      context_length=CONTEXT_LENGTH, horizon=HORIZON))

if USE_FOUNDATION_MODELS:
    cfg = dc.replace(cfg,
        price_encoder=dc.replace(cfg.price_encoder, backend='moirai',
                                 context_length=CONTEXT_LENGTH, latent_dim=LATENT_DIM),
        predictor=dc.replace(cfg.predictor, backend='lm', latent_dim=LATENT_DIM))
    device, batch = ('cuda' if torch.cuda.is_available() else 'cpu'), 256
else:
    cfg = dc.replace(cfg,
        price_encoder=dc.replace(cfg.price_encoder, backend='transformer',
                                 context_length=CONTEXT_LENGTH, latent_dim=LATENT_DIM,
                                 freeze_backbone=False),
        predictor=dc.replace(cfg.predictor, backend='transformer', latent_dim=LATENT_DIM))
    device, batch = 'cpu', 64

cfg = dc.replace(cfg, train=dc.replace(cfg.train, device=device, batch_size=batch,
                                       max_steps=2000, warmup_proj_steps=500, num_workers=2))
print(f"backend(price)={cfg.price_encoder.backend}  backend(pred)={cfg.predictor.backend}"
      f"  reg={cfg.regularizer}  device={cfg.train.device}  batch={cfg.train.batch_size}")

## 4 · Data

Each ticker parquet is converted to per-timestep log-return features and sliced into
`context` `[L, F]` / `target` `[H, F]` windows. Normalization is causal (context-window
stats only); the train/val split is chronological per ticker.

In [ ]:
from jepa_quant import build_dataloaders

train_loader, val_loader = build_dataloaders(cfg)
print(f"train windows: {len(train_loader.dataset):,}   val windows: {len(val_loader.dataset):,}")

b = next(iter(train_loader))
print('context', tuple(b['context'].shape), ' target', tuple(b['target'].shape))

## 5 · Build components & train

Phase 1 (steps 1–500): projection layers only. Phase 2 (500+): LoRA / encoder
adapters join. The EMA target encoder updates after every optimizer step and never
receives gradients. If `regularizer='codebook'`, the codebook is k-means-initialized
from a frozen forward pass before training.

In [ ]:
from jepa_quant import build_components, JEPATrainer

components = build_components(cfg)
trainer    = JEPATrainer(cfg, components, train_loader, val_loader)
history    = trainer.train()

## 6 · Diagnostics

Watch `z_price` per-dim std as a variance-collapse early warning (VICReg), or codebook
utilization (codebook — reduce `λ_commit` if <50%).

In [ ]:
import matplotlib.pyplot as plt

steps = [h['step'] for h in history]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

ax[0].plot(steps, [h['jepa'] for h in history], label='JEPA (1 - cos)', lw=2)
if any('reg/v' in h for h in history):
    ax[0].plot(steps, [h.get('reg/v') for h in history], label='VICReg V', alpha=.7)
    ax[0].plot(steps, [h.get('reg/c') for h in history], label='VICReg C', alpha=.7)
if any('reg/commit' in h for h in history):
    ax[0].plot(steps, [h.get('reg/commit') for h in history], label='commit', alpha=.7)
ax[0].set_title('Loss breakdown'); ax[0].set_xlabel('step'); ax[0].legend()

if any('reg/z_std_mean' in h for h in history):
    ax[1].plot(steps, [h.get('reg/z_std_mean') for h in history], color='crimson')
    ax[1].set_title('z_price mean per-dim std  (→ 0 means collapse)')
elif any('reg/codebook_util' in h for h in history):
    ax[1].plot(steps, [h.get('reg/codebook_util') for h in history], color='teal')
    ax[1].axhline(0.5, ls='--', c='gray'); ax[1].set_title('codebook utilization')
ax[1].set_xlabel('step')
plt.tight_layout(); plt.show()

## 7 · Comparing anti-collapse methods

Because the regularizer is selected by name, comparing methods is a one-line change —
set `REGULARIZER = 'codebook'` in the config cell and re-run from there. Both methods
share the exact same encoders, predictor, data, and training loop, so the JEPA loss
curves and `val_jepa` are directly comparable. Register new methods with
`@register_regularizer('name')` in `src/jepa_quant/regularization/` and they appear in
`registered_regularizers()` automatically.